# 03 — Train YOLO26 on BDD100K (Vehicle-Only Detection)

**Goal:** Train a YOLO26-S detection model on BDD100K for vehicle-only classes (car, truck, bus, motorcycle, bicycle) and save the checkpoint for downstream use in the joint pipeline (Notebook 08).

Steps:
1. Setup environment and reproducibility
2. Extract and verify the dataset
3. Train YOLO26-S with SGD (stable fine-tuning)
4. Validate and report mAP
5. Save the best checkpoint to Google Drive

## 1 — Setup

In [2]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.8 MB/s eta 0:00:00


In [ ]:
import os, random, shutil
import numpy as np
import torch
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Paths
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = "/content/bdd100k_yolo"
WEIGHTS_DIR = f"{ECOCAR_ROOT}/weights"
DET_CKPT_NAME = "bdd100k_yolo26s_vehicle_best.pt"  # vehicle-only checkpoint

os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"ECOCAR_ROOT : {ECOCAR_ROOT}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"WEIGHTS_DIR : {WEIGHTS_DIR}")
print(f"Checkpoint  : {DET_CKPT_NAME}")
print(f"CUDA        : {torch.cuda.is_available()}")
print(f"Classes     : car, truck, bus, motorcycle, bicycle (nc=5)")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

## 2 — Dataset

In [4]:
import tarfile

TAR_PATH = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo_nb02.tar")
DATASET_YAML = f"{DATASET_DIR}/bdd100k_yolo.yaml"

# Extract dataset from tar if not already present
if not os.path.isdir(os.path.join(DATASET_DIR, "images", "val")):
    assert os.path.isfile(TAR_PATH), (
        f"Tar archive not found: {TAR_PATH}\n"
        f"Run Notebook 02 first to create the YOLO dataset."
    )
    os.makedirs(DATASET_DIR, exist_ok=True)
    print(f"Extracting {TAR_PATH} ...")
    with tarfile.open(TAR_PATH, "r") as tar:
        tar.extractall(DATASET_DIR)
    print("Extraction complete.")
else:
    print(f"Dataset already exists: {DATASET_DIR}")

# Verify dataset YAML
assert os.path.isfile(DATASET_YAML), f"Dataset YAML not found: {DATASET_YAML}"
print(f"Dataset YAML verified: {DATASET_YAML}")

# Quick sanity check
for split in ["train", "val"]:
    img_dir = os.path.join(DATASET_DIR, "images", split)
    lbl_dir = os.path.join(DATASET_DIR, "labels", split)
    n_img = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"  {split}: {n_img} images, {n_lbl} labels")

Extracting /content/drive/MyDrive/EcoCAR/datasets/bdd100k_yolo_nb02.tar ...


/tmp/ipykernel_1687/3511493826.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATASET_DIR)


Extraction complete.
Dataset YAML verified: /content/bdd100k_yolo/bdd100k_yolo.yaml
  train: 70000 images, 70000 labels
  val: 10000 images, 10000 labels


## 3 — Train

In [5]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

# Fine-tuning config: lower LR + SGD for stable transfer from COCO pretrained weights.
# AdamW at lr0=0.01 caused validation collapse around epoch 14-15.
results = model.train(
    data=DATASET_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.002,                # lower LR for fine-tuning (was 0.01)
    lrf=0.01,                 # cosine schedule → final LR = lr0 * 0.01
    warmup_epochs=3,
    optimizer="SGD",          # SGD is more stable for YOLO fine-tuning
    amp=True,
    patience=10,
    project=f"{ECOCAR_ROOT}/training_runs",
    name="bdd100k_yolo26s",
    exist_ok=True,
    cache=False,
    verbose=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bdd100k_yolo/bdd100k_yolo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv

## 4 — Validate

In [6]:
from ultralytics import YOLO

# Load the best checkpoint from training
BEST_PT = f"{ECOCAR_ROOT}/training_runs/bdd100k_yolo26s/weights/best.pt"
assert os.path.isfile(BEST_PT), f"best.pt not found: {BEST_PT}"

model = YOLO(BEST_PT)
metrics = model.val(data=DATASET_YAML)

map50 = metrics.box.map50 * 100
map50_95 = metrics.box.map * 100

print(f"mAP@50:    {map50:.1f}%")
print(f"mAP@50-95: {map50_95:.1f}%")

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO26s summary (fused): 122 layers, 9,469,050 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4060.7±1827.9 MB/s, size: 54.7 KB)
val: Scanning /content/bdd100k_yolo/labels/val.cache... 10000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10000/10000 5.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 625/625 34.8it/s 18.0s
                   all      10000     184119      0.735      0.482      0.537       0.31
                person       3220      13265      0.729      0.514      0.605      0.303
                 rider        515        649       0.68      0.357       0.42      0.209
                   car       9879     102540      0.789      0.698      0.782      0.484
                 truck       2689       4247      0.645      0.565      0.609      0.

## 5 — Save Checkpoint

In [ ]:
import shutil

dst = os.path.join(WEIGHTS_DIR, DET_CKPT_NAME)
shutil.copy2(BEST_PT, dst)
assert os.path.isfile(dst), f"Failed to copy checkpoint to {dst}"

print("=" * 46)
print(" VEHICLE DETECTION TRAINING COMPLETE")
print(f" Classes: car, truck, bus, motorcycle, bicycle")
print(f" mAP@50:    {map50:.1f}%")
print(f" mAP@50-95: {map50_95:.1f}%")
print(f" Checkpoint: {dst}")
print()
print(" Use this checkpoint in Notebook 08 as:")
print("   warm_start for joint vehicle+lane training")
print("=" * 46)